In [1]:
# セル1: 必要モジュールとデータロード
import numpy as np
import pickle
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 埋め込み行列と SST データのロード
embedding_matrix = np.load('70_embeddings.npy')
with open('71_sst_data.pkl', 'rb') as f:
    data = pickle.load(f)
train_data = data['train']
dev_data   = data['dev']


In [2]:
# セル2: collate 関数＋BoWClassifier 定義
def collate(batch):
    # 長い順ソート
    batch = sorted(batch, key=lambda x: x['input_ids'].size(0), reverse=True)
    xs = [ex['input_ids'] for ex in batch]
    ys = [ex['label']     for ex in batch]
    x_pad = pad_sequence(xs, batch_first=True, padding_value=0)
    y_cat = torch.cat(ys).view(-1,1)
    return x_pad, y_cat

class BoWClassifier(nn.Module):
    def __init__(self, emb_matrix):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(emb_matrix, dtype=torch.float),
            freeze=False,  # emb は後で固定します
            padding_idx=0
        )
        emb_dim = emb_matrix.shape[1]
        self.weight = nn.Parameter(torch.zeros(emb_dim))
        self.bias   = nn.Parameter(torch.zeros(1))

    def forward(self, input_ids):
        emb  = self.embedding(input_ids)                     # (B,L,D)
        mask = (input_ids!=0).unsqueeze(-1).float()          # (B,L,1)
        s    = (emb*mask).sum(dim=1)                         # (B,D)
        l    = mask.sum(dim=1).clamp(min=1)                  # (B,1)
        avg  = s / l                                         # (B,D)
        logit= avg.matmul(self.weight) + self.bias           # (B,)
        return torch.sigmoid(logit)


In [3]:
# セル3: モデル学習（埋め込み固定）＆開発セット評価
model     = BoWClassifier(embedding_matrix).to(device)
# 埋め込みだけ固定
model.embedding.weight.requires_grad = False

optimizer = torch.optim.Adam([model.weight, model.bias], lr=1e-3)
criterion = nn.BCELoss()

loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate)

# 学習ループ
model.train()
for epoch in range(3):
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y.squeeze())
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch} done")

# 評価
model.eval()
correct = 0
with torch.no_grad():
    for i in range(0, len(dev_data), 32):
        batch = dev_data[i:i+32]
        x, y = collate(batch)
        x, y = x.to(device), y.to(device)
        pred = model(x).round()
        correct += (pred.squeeze() == y.squeeze()).sum().item()
acc = correct / len(dev_data)
print(f"Dev Accuracy (GPU): {acc:.4f}")


Epoch 0 done
Epoch 1 done
Epoch 2 done
Dev Accuracy (GPU): 0.7810
